# SOXX NPORT-P filing index 离线元数据核查

在 13 份已由 NAS 定点恢复的 SEC 索引页中核对 SOXX series、报告期、接受显示时间和原 XML 文件链接；不读原 XML 数值。


In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2019_2022_nport_filing_metadata_20261023';plan=json.loads((card/'plan.json').read_text());captured=json.loads((card/'capture_receipts.json').read_text());transport=json.loads((card/'transport_result.json').read_text())
assert transport['status']=='thirteen_exact_NPORT_index_pages_archived_parse_pending' and transport['nas_remote_and_scoped_restore_verified']
assert len(captured)==len(plan['exact_index_pages'])==13
by_accession={x['accession']:x for x in captured};raw=root/'data/snapshots/stage1_soxx_2019_2022_nport_filing_metadata_20261023'
rows=[];errors=[]
for item in plan['exact_index_pages']:
 accession=item['accession'];receipt=by_accession[accession];path=raw/(accession+'.html')
 if hashlib.sha256(path.read_bytes()).hexdigest()!=receipt['raw_sha256']:errors.append((accession,'raw hash drift'));continue
 soup=BeautifulSoup(path.read_bytes(),'html.parser')
 try:
  secnum=soup.select_one('#secNum');assert secnum and accession in secnum.get_text(' ',strip=True)
  formname=soup.select_one('#formName');assert formname and 'NPORT-P' in formname.get_text(' ',strip=True)
  info={}
  for group in soup.select('div.formContent div.formGrouping'):
   children=group.find_all('div',recursive=False)
   for i,node in enumerate(children[:-1]):
    if 'infoHead' in (node.get('class') or []):info[node.get_text(' ',strip=True)]=children[i+1].get_text(' ',strip=True)
  filed=info.get('Filing Date');accepted=info.get('Accepted');period=info.get('Period of Report')
  assert filed==item['filed_from_series_search'] and re.fullmatch(r'20\d{2}-\d{2}-\d{2}',filed)
  assert accepted and re.fullmatch(r'20\d{2}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}',accepted)
  assert period and re.fullmatch(r'20\d{2}-\d{2}-\d{2}',period) and period<=filed
  div=soup.select_one('#seriesDiv');assert div
  text=div.get_text(' ',strip=True)
  assert 'S000004354' in text and 'C000012084' in text and re.search(r'\bSOXX\b',text) and '0001100663' in text
  docs=[]
  for tr in soup.select('table.tableFile tr'):
   cells=tr.find_all('td',recursive=False)
   if len(cells)<5:continue
   link=cells[2].find('a');
   if link:docs.append({'document':link.get_text(' ',strip=True),'href':link.get('href'),'type':cells[3].get_text(' ',strip=True),'size_text':cells[4].get_text(' ',strip=True)})
  xml=[x for x in docs if x['type']=='NPORT-P' and x['document'].lower().endswith('.xml') and 'xsl' not in x['href'].lower()]
  assert len(xml)==1
  rows.append({'accession':accession,'form':'NPORT-P','filed':filed,'accepted_display':accepted,'accepted_display_timezone':'SEC_page_unverified','report_period':period,'series_id':'S000004354','class_id':'C000012084','ticker':'SOXX','primary_xml':xml[0],'exhibits':[x for x in docs if x['type'] in ('NPORT-EX','NPORT-EX/A')],'source_index_sha256':receipt['raw_sha256']})
 except Exception as exc:errors.append((accession,type(exc).__name__+':'+str(exc)[:150]))
result={'status':'thirteen_series_specific_NPORT_index_metadata_qualified_bodies_pending' if not errors and len(rows)==13 else 'NPORT_index_metadata_mismatch_stop','source_plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'source_archive_sha256':transport['nas_archive_sha256'],'source_NAS_scoped_restore_verified':True,'expected_pages':13,'parsed_pages':len(rows),'errors':errors,'rows':rows,'accepted_clock_timezone_qualified':False,'original_holdings_values_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'metadata_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','expected_pages','parsed_pages','errors')})


{'status': 'thirteen_series_specific_NPORT_index_metadata_qualified_bodies_pending', 'expected_pages': 13, 'parsed_pages': 13, 'errors': []}


## 中文结论

13/13 份原 filing index 的 accession、申报日、报告期和 SOXX series/class 身份与冻结入口匹配，原 XML 链接逐一保存；最后一个报告期 2022-09-30 在 2022-11-28 申报。SEC 页面接受时间按原文保留，时区未单独认证。只验证索引，不读取持仓数值或收益。
